In [1]:
# ============================================================
# CELL 29.1 — PHASE 2 DATASET AUDIT
# Dataset manifest, labels, exact duplicates, patient IDs,
# and stratified train/validation split
# ============================================================

import os
import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INCXR_ROOT = Path("/kaggle/input/datasets/arjav007/in-cxr-dataset-png")

VINDR_ROOT = Path(
    "/kaggle/input/competitions/"
    "vinbigdata-chest-xray-abnormalities-detection"
)

VINDR_CSV = VINDR_ROOT / "train.csv"

OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

print("=" * 70)
print("LUNGVISION AI — PHASE 2 DATASET AUDIT")
print("=" * 70)

assert INCXR_ROOT.exists(), f"IN-CXR not found: {INCXR_ROOT}"
assert VINDR_ROOT.exists(), f"VinDr not found: {VINDR_ROOT}"
assert VINDR_CSV.exists(), f"VinDr train.csv not found: {VINDR_CSV}"

# ------------------------------------------------------------
# 2. IN-CXR IMAGE DISCOVERY AND LABEL EXTRACTION
# ------------------------------------------------------------

print("\n[1/8] Auditing IN-CXR dataset...")

incxr_records = []

image_extensions = {".png", ".jpg", ".jpeg"}

incxr_files = [
    p for p in INCXR_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print("Discovered IN-CXR images:", len(incxr_files))

for path in tqdm(incxr_files, desc="IN-CXR manifest"):

    folder_parts = [part.lower() for part in path.parts]

    label = None

    if any(part in {"normal", "normal_images"} for part in folder_parts):
        label = 0

    elif any(
        part in {"abnormal", "abnormal_images", "pneumonia"}
        for part in folder_parts
    ):
        label = 1

    if label is None:
        continue

    incxr_records.append({
        "image_id": path.stem,
        "source": "IN-CXR",
        "label": label,
        "label_name": "Abnormal" if label == 1 else "Normal",
        "file_path": str(path),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "patient_id": None
    })

incxr_df = pd.DataFrame(incxr_records)

if len(incxr_df) == 0:
    raise RuntimeError(
        "No IN-CXR images were assigned labels. "
        "Inspect the dataset's folder structure before continuing."
    )

print("\nIN-CXR image count:", len(incxr_df))
print("\nIN-CXR label distribution:")
print(incxr_df["label_name"].value_counts())

# ------------------------------------------------------------
# 3. VINDR LABEL EXTRACTION
# ------------------------------------------------------------

print("\n[2/8] Auditing VinDr annotations...")

vindr_annotations = pd.read_csv(VINDR_CSV)

required_columns = {
    "image_id",
    "class_id",
    "class_name",
    "rad_id"
}

missing_columns = required_columns - set(vindr_annotations.columns)

if missing_columns:
    raise ValueError(f"Missing VinDr columns: {missing_columns}")

vindr_annotations["class_id"] = pd.to_numeric(
    vindr_annotations["class_id"],
    errors="coerce"
)

vindr_annotations["reader_abnormal"] = (
    vindr_annotations["class_id"] != 14
).astype(int)

# One binary decision per image per radiologist.
reader_labels = (
    vindr_annotations
    .groupby(["image_id", "rad_id"], as_index=False)
    .agg(reader_label=("reader_abnormal", "max"))
)

# One binary target per image.
vindr_labels = (
    reader_labels
    .groupby("image_id", as_index=False)
    .agg(
        label=("reader_label", "max"),
        reader_count=("rad_id", "count"),
        reader_positive_count=("reader_label", "sum"),
        reader_negative_count=(
            "reader_label",
            lambda x: int((x == 0).sum())
        )
    )
)

vindr_labels["label_name"] = np.where(
    vindr_labels["label"] == 1,
    "Abnormal",
    "Normal"
)

vindr_labels["reader_unanimous"] = (
    (vindr_labels["reader_positive_count"] == 0) |
    (vindr_labels["reader_negative_count"] == 0)
)

print("VinDr annotation rows:", len(vindr_annotations))
print("VinDr unique images:", len(vindr_labels))
print("Reader count distribution:")
print(vindr_labels["reader_count"].value_counts().sort_index())

print("\nBinary label distribution:")
print(vindr_labels["label_name"].value_counts())

print(
    "\nBinary reader unanimity:",
    f"{vindr_labels['reader_unanimous'].mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 4. VINDR DICOM DISCOVERY AND MATCHING
# ------------------------------------------------------------

print("\n[3/8] Discovering VinDr DICOM files...")

dicom_files = []

for path in VINDR_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".dicom", ".dcm"}:
        dicom_files.append(path)

print("Discovered DICOM files:", len(dicom_files))

dicom_map = {}

for path in dicom_files:
    dicom_map[path.stem.lower()] = path

vindr_records = []
missing_dicom = []

try:
    import pydicom
except ImportError:
    raise ImportError("pydicom is required. Run: !pip install pydicom")

for row in tqdm(
    vindr_labels.itertuples(index=False),
    total=len(vindr_labels),
    desc="VinDr manifest"
):

    image_id = str(row.image_id)
    path = dicom_map.get(image_id.lower())

    if path is None:
        missing_dicom.append(image_id)
        continue

    patient_id = None

    try:
        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            force=True
        )

        value = getattr(ds, "PatientID", None)

        if value is not None and str(value).strip():
            patient_id = str(value).strip()

    except Exception:
        pass

    vindr_records.append({
        "image_id": image_id,
        "source": "VinDr",
        "label": int(row.label),
        "label_name": row.label_name,
        "file_path": str(path),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "patient_id": patient_id,
        "reader_count": int(row.reader_count),
        "reader_unanimous": bool(row.reader_unanimous)
    })

vindr_df = pd.DataFrame(vindr_records)

print("\nMatched VinDr DICOM files:", len(vindr_df))
print("Missing DICOM files:", len(missing_dicom))

if len(vindr_df) == 0:
    raise RuntimeError("No VinDr DICOM files matched the annotation CSV.")

if missing_dicom:
    pd.DataFrame({"image_id": missing_dicom}).to_csv(
        OUTPUT_DIR / "vindr_missing_dicom.csv",
        index=False
    )

# ------------------------------------------------------------
# 5. COMBINE DATASET MANIFEST
# ------------------------------------------------------------

print("\n[4/8] Creating combined manifest...")

manifest = pd.concat(
    [incxr_df, vindr_df],
    ignore_index=True,
    sort=False
)

manifest["label"] = manifest["label"].astype(int)

print("\nCombined image count:", len(manifest))
print("\nSource distribution:")
print(manifest["source"].value_counts())

print("\nSource-wise binary labels:")
print(
    pd.crosstab(
        manifest["source"],
        manifest["label_name"]
    )
)

# ------------------------------------------------------------
# 6. EXACT SHA256 DUPLICATE AUDIT
# ------------------------------------------------------------

print("\n[5/8] Computing exact SHA256 hashes...")

def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


hashes = []

for path in tqdm(
    manifest["file_path"],
    total=len(manifest),
    desc="SHA256 audit"
):
    try:
        hashes.append(calculate_sha256(path))
    except Exception:
        hashes.append(None)

manifest["sha256"] = hashes

hash_counts = manifest["sha256"].value_counts(dropna=True)

duplicate_hashes = hash_counts[hash_counts > 1].index.tolist()

duplicate_records = manifest[
    manifest["sha256"].isin(duplicate_hashes)
].copy()

manifest["exact_duplicate"] = (
    manifest["sha256"].isin(duplicate_hashes)
)

manifest["duplicate_group"] = manifest["sha256"].where(
    manifest["exact_duplicate"],
    None
)

print("\nFiles with duplicate SHA256:", len(duplicate_records))
print("Exact duplicate groups:", len(duplicate_hashes))

if len(duplicate_records):
    print("\nDuplicate records by source:")
    print(duplicate_records["source"].value_counts())

    print("\nDuplicate records by label:")
    print(duplicate_records["label_name"].value_counts())

    print("\nCross-source duplicate groups:")

    cross_source_groups = (
        duplicate_records.groupby("sha256")["source"]
        .nunique()
    )

    print(int((cross_source_groups > 1).sum()))

# Conservative quarantine: remove every member of an exact
# duplicate group from development. Original files remain untouched.
manifest["quarantined"] = manifest["exact_duplicate"]

duplicate_records.to_csv(
    OUTPUT_DIR / "exact_duplicate_quarantine.csv",
    index=False
)

clean_manifest = manifest[
    ~manifest["quarantined"]
].copy()

print("\nClean development image count:", len(clean_manifest))
print("Quarantined image count:", int(manifest["quarantined"].sum()))

# ------------------------------------------------------------
# 7. PATIENT IDENTIFIER AUDIT
# ------------------------------------------------------------

print("\n[6/8] Auditing patient identifiers...")

patient_audit = (
    manifest.groupby("source")
    .agg(
        total_images=("image_id", "count"),
        images_with_patient_id=(
            "patient_id",
            lambda x: int(x.notna().sum())
        ),
        unique_patient_ids=(
            "patient_id",
            lambda x: int(x.dropna().nunique())
        )
    )
    .reset_index()
)

patient_audit["images_without_patient_id"] = (
    patient_audit["total_images"] -
    patient_audit["images_with_patient_id"]
)

print(patient_audit.to_string(index=False))

# ------------------------------------------------------------
# 8. STRATIFIED TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------

print("\n[7/8] Creating stratified train/validation splits...")

split_frames = []

for source in sorted(clean_manifest["source"].unique()):

    source_df = clean_manifest[
        clean_manifest["source"] == source
    ].copy()

    train_df, val_df = train_test_split(
        source_df,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=source_df["label"]
    )

    train_df["split"] = "train"
    val_df["split"] = "validation"

    split_frames.extend([train_df, val_df])

    print(f"\n{source}")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))

split_manifest = pd.concat(
    split_frames,
    ignore_index=True
)

# Check image-level split integrity.
train_hashes = set(
    split_manifest.loc[
        split_manifest["split"] == "train",
        "sha256"
    ].dropna()
)

val_hashes = set(
    split_manifest.loc[
        split_manifest["split"] == "validation",
        "sha256"
    ].dropna()
)

hash_overlap = train_hashes.intersection(val_hashes)

train_ids = set(
    split_manifest.loc[
        split_manifest["split"] == "train",
        ["source", "image_id"]
    ].apply(tuple, axis=1)
)

val_ids = set(
    split_manifest.loc[
        split_manifest["split"] == "validation",
        ["source", "image_id"]
    ].apply(tuple, axis=1)
)

image_id_overlap = train_ids.intersection(val_ids)

print("\nTrain/validation SHA256 overlap:", len(hash_overlap))
print("Train/validation image ID overlap:", len(image_id_overlap))

if hash_overlap or image_id_overlap:
    raise RuntimeError("Train/validation leakage detected.")

# ------------------------------------------------------------
# 9. FINAL SUMMARIES
# ------------------------------------------------------------

print("\n[8/8] Saving audit artifacts...")

source_summary = (
    split_manifest.groupby(["source", "split", "label_name"])
    .size()
    .reset_index(name="image_count")
)

source_summary.to_csv(
    OUTPUT_DIR / "source_split_label_summary.csv",
    index=False
)

manifest.to_csv(
    OUTPUT_DIR / "phase2_full_manifest.csv",
    index=False
)

clean_manifest.to_csv(
    OUTPUT_DIR / "phase2_clean_manifest.csv",
    index=False
)

split_manifest.to_csv(
    OUTPUT_DIR / "phase2_train_validation_split.csv",
    index=False
)

patient_audit.to_csv(
    OUTPUT_DIR / "patient_id_audit.csv",
    index=False
)

# Store metadata.
metadata = {
    "audit_name": "LungVision AI Phase 2 Dataset Audit",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_state": RANDOM_STATE,
    "validation_size": VALIDATION_SIZE,
    "incxr_root": str(INCXR_ROOT),
    "vindr_root": str(VINDR_ROOT),
    "vindr_csv": str(VINDR_CSV),
    "incxr_images_discovered": int(len(incxr_files)),
    "incxr_images_labeled": int(len(incxr_df)),
    "vindr_annotation_rows": int(len(vindr_annotations)),
    "vindr_unique_annotated_images": int(len(vindr_labels)),
    "vindr_dicom_matched": int(len(vindr_df)),
    "vindr_dicom_missing": int(len(missing_dicom)),
    "total_manifest_images": int(len(manifest)),
    "exact_duplicate_groups": int(len(duplicate_hashes)),
    "exact_duplicate_records": int(len(duplicate_records)),
    "quarantined_records": int(manifest["quarantined"].sum()),
    "clean_development_images": int(len(clean_manifest)),
    "train_images": int((split_manifest["split"] == "train").sum()),
    "validation_images": int(
        (split_manifest["split"] == "validation").sum()
    ),
    "train_validation_sha256_overlap": int(len(hash_overlap)),
    "train_validation_image_id_overlap": int(len(image_id_overlap)),
    "patient_audit": patient_audit.to_dict(orient="records"),
    "limitations": [
        "Image-level split only; patient-disjoint validation is not certified.",
        "Missing patient identifiers prevent reliable patient-level grouping.",
        "VinDr binary labels are derived from the three-reader training annotations.",
        "Exact SHA256 detects byte-identical files, not visually similar images.",
        "All members of exact duplicate groups are quarantined conservatively."
    ]
}

with open(OUTPUT_DIR / "phase2_audit_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("PHASE 2 DATASET AUDIT COMPLETED")
print("=" * 70)

print("\nFINAL COUNTS")
print("IN-CXR labeled images:", len(incxr_df))
print("VinDr matched images:", len(vindr_df))
print("Combined images:", len(manifest))
print("Exact duplicate records:", len(duplicate_records))
print("Quarantined records:", int(manifest["quarantined"].sum()))
print("Clean development images:", len(clean_manifest))
print("Training images:", len(split_manifest[split_manifest["split"] == "train"]))
print("Validation images:", len(split_manifest[split_manifest["split"] == "validation"]))

print("\nPATIENT ID LIMITATION")
print("Patient-disjoint split certified: NO")

print("\nOUTPUT DIRECTORY")
print(OUTPUT_DIR)

print("\nGENERATED FILES")
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(f"{file.name} ({file.stat().st_size / 1024:.1f} KB)")

LUNGVISION AI — PHASE 2 DATASET AUDIT

[1/8] Auditing IN-CXR dataset...
Discovered IN-CXR images: 12600


IN-CXR manifest:   0%|          | 0/12600 [00:00<?, ?it/s]


IN-CXR image count: 12600

IN-CXR label distribution:
label_name
Abnormal    6300
Normal      6300
Name: count, dtype: int64

[2/8] Auditing VinDr annotations...
VinDr annotation rows: 67914
VinDr unique images: 15000
Reader count distribution:
reader_count
3    15000
Name: count, dtype: int64

Binary label distribution:
label_name
Normal      10606
Abnormal     4394
Name: count, dtype: int64

Binary reader unanimity: 100.00%

[3/8] Discovering VinDr DICOM files...
Discovered DICOM files: 18000


VinDr manifest:   0%|          | 0/15000 [00:00<?, ?it/s]


Matched VinDr DICOM files: 15000
Missing DICOM files: 0

[4/8] Creating combined manifest...

Combined image count: 27600

Source distribution:
source
VinDr     15000
IN-CXR    12600
Name: count, dtype: int64

Source-wise binary labels:
label_name  Abnormal  Normal
source                      
IN-CXR          6300    6300
VinDr           4394   10606

[5/8] Computing exact SHA256 hashes...


SHA256 audit:   0%|          | 0/27600 [00:00<?, ?it/s]


Files with duplicate SHA256: 86
Exact duplicate groups: 43

Duplicate records by source:
source
IN-CXR    86
Name: count, dtype: int64

Duplicate records by label:
label_name
Abnormal    80
Normal       6
Name: count, dtype: int64

Cross-source duplicate groups:
0

Clean development image count: 27514
Quarantined image count: 86

[6/8] Auditing patient identifiers...
source  total_images  images_with_patient_id  unique_patient_ids  images_without_patient_id
IN-CXR         12600                       0                   0                      12600
 VinDr         15000                       0                   0                      15000

[7/8] Creating stratified train/validation splits...

IN-CXR
Train: 10011
Validation: 2503

VinDr
Train: 12000
Validation: 3000

Train/validation SHA256 overlap: 0
Train/validation image ID overlap: 0

[8/8] Saving audit artifacts...

PHASE 2 DATASET AUDIT COMPLETED

FINAL COUNTS
IN-CXR labeled images: 12600
VinDr matched images: 15000
Combined image

In [2]:
# ============================================================
# CELL 29.2 — PHASE 2 DATASET PROFILE
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_clean_manifest.csv"
)

split_df = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_dataset_profile")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("LUNGVISION AI — PHASE 2 DATASET PROFILE")
print("=" * 70)

# 1. Source-level dataset profile
source_profile = (
    manifest.groupby("source")
    .agg(
        total_images=("image_id", "count"),
        normal_images=("label", lambda x: int((x == 0).sum())),
        abnormal_images=("label", lambda x: int((x == 1).sum())),
        unique_image_ids=("image_id", "nunique")
    )
    .reset_index()
)

source_profile["abnormal_prevalence"] = (
    source_profile["abnormal_images"] /
    source_profile["total_images"]
)

print("\nSOURCE PROFILE")
print(source_profile.to_string(index=False))

# 2. Train-validation distribution
split_profile = (
    split_df.groupby(["source", "split", "label_name"])
    .size()
    .reset_index(name="image_count")
)

print("\nSPLIT DISTRIBUTION")
print(split_profile.to_string(index=False))

# 3. Class prevalence by source and split
prevalence = (
    split_df.groupby(["source", "split"])
    .agg(
        total_images=("label", "size"),
        abnormal_images=("label", "sum"),
        normal_images=("label", lambda x: int((x == 0).sum()))
    )
    .reset_index()
)

prevalence["abnormal_prevalence"] = (
    prevalence["abnormal_images"] /
    prevalence["total_images"]
)

print("\nABNORMAL PREVALENCE")
print(prevalence.to_string(index=False))

# 4. Class weights
class_weights = {}

for source in sorted(split_df["source"].unique()):

    train = split_df[
        (split_df["source"] == source) &
        (split_df["split"] == "train")
    ]

    counts = train["label"].value_counts().to_dict()
    total = len(train)

    weights = {
        0: total / (2 * counts.get(0, 1)),
        1: total / (2 * counts.get(1, 1))
    }

    class_weights[source] = weights

    print(f"\n{source} class weights:")
    print("Normal:", round(weights[0], 4))
    print("Abnormal:", round(weights[1], 4))

# 5. Image dimension and format audit
try:
    from PIL import Image
except ImportError:
    raise ImportError("Pillow is required.")

print("\nIMAGE DIMENSION AUDIT")

sample_size = min(500, len(manifest))

sample_df = manifest.sample(
    n=sample_size,
    random_state=42
)

dimension_records = []

for row in sample_df.itertuples(index=False):

    record = {
        "source": row.source,
        "label": row.label_name,
        "extension": row.extension,
        "width": None,
        "height": None,
        "mode": None,
        "error": None
    }

    if row.source == "IN-CXR":
        try:
            with Image.open(row.file_path) as image:
                record["width"], record["height"] = image.size
                record["mode"] = image.mode

        except Exception as error:
            record["error"] = str(error)

    dimension_records.append(record)

dimension_df = pd.DataFrame(dimension_records)

print("\nIN-CXR dimension distribution:")
print(
    dimension_df[
        dimension_df["source"] == "IN-CXR"
    ][["width", "height"]].describe()
)

print("\nIN-CXR image modes:")
print(
    dimension_df[
        dimension_df["source"] == "IN-CXR"
    ]["mode"].value_counts()
)

# 6. Missing metadata
metadata_audit = (
    manifest.groupby("source")
    .agg(
        missing_patient_ids=("patient_id", lambda x: int(x.isna().sum())),
        missing_hashes=("sha256", lambda x: int(x.isna().sum())),
        missing_file_paths=("file_path", lambda x: int(x.isna().sum()))
    )
    .reset_index()
)

print("\nMETADATA AUDIT")
print(metadata_audit.to_string(index=False))

# 7. Save outputs
source_profile.to_csv(
    OUTPUT_DIR / "source_profile.csv",
    index=False
)

split_profile.to_csv(
    OUTPUT_DIR / "split_profile.csv",
    index=False
)

prevalence.to_csv(
    OUTPUT_DIR / "prevalence_profile.csv",
    index=False
)

dimension_df.to_csv(
    OUTPUT_DIR / "image_dimension_sample.csv",
    index=False
)

metadata_audit.to_csv(
    OUTPUT_DIR / "metadata_audit.csv",
    index=False
)

with open(OUTPUT_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=4)

print("\n" + "=" * 70)
print("CELL 29.2 COMPLETED")
print("=" * 70)
print("Output directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

LUNGVISION AI — PHASE 2 DATASET PROFILE

SOURCE PROFILE
source  total_images  normal_images  abnormal_images  unique_image_ids  abnormal_prevalence
IN-CXR         12514           6294             6220             12514             0.497043
 VinDr         15000          10606             4394             15000             0.292933

SPLIT DISTRIBUTION
source      split label_name  image_count
IN-CXR      train   Abnormal         4976
IN-CXR      train     Normal         5035
IN-CXR validation   Abnormal         1244
IN-CXR validation     Normal         1259
 VinDr      train   Abnormal         3515
 VinDr      train     Normal         8485
 VinDr validation   Abnormal          879
 VinDr validation     Normal         2121

ABNORMAL PREVALENCE
source      split  total_images  abnormal_images  normal_images  abnormal_prevalence
IN-CXR      train         10011             4976           5035             0.497053
IN-CXR validation          2503             1244           1259             0.4

In [3]:
# ============================================================
# CELL 29.3 — PHASE 2 PIXEL DOMAIN AUDIT
# IN-CXR PNG vs VinDr DICOM
# ============================================================

import numpy as np
import pandas as pd
import pydicom
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from pydicom.pixel_data_handlers.util import (
    apply_modality_lut,
    apply_voi_lut
)

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_pixel_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

RANDOM_STATE = 42
SAMPLES_PER_GROUP = 40

print("=" * 70)
print("LUNGVISION AI — PIXEL DOMAIN AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. STRATIFIED SAMPLE
# ------------------------------------------------------------

sample_groups = []

for (source, split, label), group in manifest.groupby(
    ["source", "split", "label"]
):
    n = min(SAMPLES_PER_GROUP, len(group))

    sample_groups.append(
        group.sample(n=n, random_state=RANDOM_STATE)
    )

sample_df = pd.concat(sample_groups, ignore_index=True)

print("Selected images:", len(sample_df))
print("\nSampling distribution:")
print(
    sample_df.groupby(
        ["source", "split", "label_name"]
    ).size()
)

# ------------------------------------------------------------
# 2. IMAGE DECODING HELPERS
# ------------------------------------------------------------

def percentile_normalize(array):
    array = np.asarray(array, dtype=np.float32)

    low, high = np.percentile(array, [1, 99])

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low = float(np.min(array))
        high = float(np.max(array))

    if high <= low:
        return np.zeros_like(array, dtype=np.float32)

    result = (array - low) / (high - low)
    return np.clip(result, 0, 1)


def read_incxr(path):
    with Image.open(path) as image:
        image = image.convert("L")
        array = np.asarray(image, dtype=np.float32)

    display = percentile_normalize(array)

    return {
        "raw": array,
        "display": display,
        "photometric": "PNG_GRAYSCALE",
        "modality_lut_applied": False,
        "voi_applied": False,
        "voi_method": "Percentile normalization",
        "monochrome1": False
    }


def read_vindr(path):
    ds = pydicom.dcmread(str(path), force=True)

    raw = ds.pixel_array.astype(np.float32)

    photometric = str(
        getattr(ds, "PhotometricInterpretation", "UNKNOWN")
    )

    monochrome1 = photometric == "MONOCHROME1"

    modality_applied = False
    voi_applied = False
    voi_method = "Percentile fallback"

    try:
        modality = apply_modality_lut(raw, ds).astype(np.float32)

        modality_applied = (
            hasattr(ds, "ModalityLUTSequence") or
            (
                hasattr(ds, "RescaleSlope") and
                hasattr(ds, "RescaleIntercept")
            )
        )

    except Exception:
        modality = raw.copy()

    has_voi = (
        hasattr(ds, "VOILUTSequence") or
        (
            hasattr(ds, "WindowCenter") and
            hasattr(ds, "WindowWidth")
        )
    )

    if has_voi:
        try:
            display_array = apply_voi_lut(
                modality,
                ds,
                index=0
            ).astype(np.float32)

            voi_applied = True
            voi_method = "DICOM VOI LUT/Window"

        except Exception:
            display_array = modality.copy()

    else:
        display_array = modality.copy()

    if monochrome1:
        display_array = (
            np.max(display_array) - display_array
        )

    display = percentile_normalize(display_array)

    return {
        "raw": raw,
        "display": display,
        "photometric": photometric,
        "modality_lut_applied": modality_applied,
        "voi_applied": voi_applied,
        "voi_method": voi_method,
        "monochrome1": monochrome1
    }


# ------------------------------------------------------------
# 3. PIXEL STATISTICS
# ------------------------------------------------------------

records = []
errors = []

for row in tqdm(
    sample_df.itertuples(index=False),
    total=len(sample_df),
    desc="Pixel audit"
):

    try:
        if row.source == "IN-CXR":
            result = read_incxr(row.file_path)
        else:
            result = read_vindr(row.file_path)

        raw = result["raw"]
        display = result["display"]

        records.append({
            "source": row.source,
            "split": row.split,
            "label": row.label,
            "label_name": row.label_name,
            "image_id": row.image_id,
            "width": int(raw.shape[1]),
            "height": int(raw.shape[0]),
            "raw_min": float(np.min(raw)),
            "raw_max": float(np.max(raw)),
            "raw_mean": float(np.mean(raw)),
            "raw_std": float(np.std(raw)),
            "display_mean_0_255": float(np.mean(display) * 255),
            "display_std_0_255": float(np.std(display) * 255),
            "near_black_fraction": float(np.mean(display <= 0.02)),
            "near_white_fraction": float(np.mean(display >= 0.98)),
            "photometric": result["photometric"],
            "modality_lut_applied": result["modality_lut_applied"],
            "voi_applied": result["voi_applied"],
            "voi_method": result["voi_method"],
            "monochrome1": result["monochrome1"]
        })

    except Exception as error:
        errors.append({
            "source": row.source,
            "image_id": row.image_id,
            "file_path": row.file_path,
            "error": str(error)
        })

pixel_df = pd.DataFrame(records)
error_df = pd.DataFrame(errors)

# ------------------------------------------------------------
# 4. DOMAIN SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PIXEL DOMAIN SUMMARY")
print("=" * 70)

summary_columns = [
    "raw_mean",
    "raw_std",
    "display_mean_0_255",
    "display_std_0_255",
    "near_black_fraction",
    "near_white_fraction"
]

domain_summary = (
    pixel_df.groupby("source")[summary_columns]
    .agg(["mean", "median", "std"])
)

print(domain_summary.to_string())

# ------------------------------------------------------------
# 5. SOURCE AND LABEL COMPARISON
# ------------------------------------------------------------

print("\nSOURCE / LABEL COMPARISON")

label_summary = (
    pixel_df.groupby(["source", "label_name"])[
        [
            "display_mean_0_255",
            "display_std_0_255",
            "near_black_fraction",
            "near_white_fraction"
        ]
    ]
    .agg(["mean", "median"])
)

print(label_summary.to_string())

# ------------------------------------------------------------
# 6. DICOM METADATA SUMMARY
# ------------------------------------------------------------

vindr_df = pixel_df[
    pixel_df["source"] == "VinDr"
]

if len(vindr_df):

    print("\nDICOM PHOTOMETRIC DISTRIBUTION")
    print(vindr_df["photometric"].value_counts())

    print("\nDICOM VOI APPLICATION")
    print(vindr_df["voi_applied"].value_counts())

    print("\nDICOM MODALITY LUT APPLICATION")
    print(vindr_df["modality_lut_applied"].value_counts())

    print("\nDICOM MONOCHROME1 COUNT")
    print(vindr_df["monochrome1"].value_counts())

    print("\nDICOM VOI METHOD")
    print(vindr_df["voi_method"].value_counts())

# ------------------------------------------------------------
# 7. VALIDATION AND SAVE
# ------------------------------------------------------------

print("\nDECODE RESULTS")
print("Selected:", len(sample_df))
print("Successfully decoded:", len(pixel_df))
print("Failed:", len(error_df))

pixel_df.to_csv(
    OUTPUT_DIR / "pixel_audit_metrics.csv",
    index=False
)

domain_summary.to_csv(
    OUTPUT_DIR / "pixel_domain_summary.csv"
)

label_summary.to_csv(
    OUTPUT_DIR / "pixel_source_label_summary.csv"
)

error_df.to_csv(
    OUTPUT_DIR / "pixel_decode_errors.csv",
    index=False
)

metadata = {
    "selected_images": int(len(sample_df)),
    "successfully_decoded": int(len(pixel_df)),
    "failed_images": int(len(error_df)),
    "samples_per_source_split_label": SAMPLES_PER_GROUP,
    "random_state": RANDOM_STATE,
    "limitations": [
        "Stratified sample, not a full dataset pixel audit.",
        "Percentile normalization is an audit display operation.",
        "Display statistics do not establish image quality or diagnostic value."
    ]
}

with open(OUTPUT_DIR / "pixel_audit_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("CELL 29.3 COMPLETED")
print("=" * 70)

print("Output directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(file.name)

LUNGVISION AI — PIXEL DOMAIN AUDIT
Selected images: 320

Sampling distribution:
source  split       label_name
IN-CXR  train       Abnormal      40
                    Normal        40
        validation  Abnormal      40
                    Normal        40
VinDr   train       Abnormal      40
                    Normal        40
        validation  Abnormal      40
                    Normal        40
dtype: int64


Pixel audit:   0%|          | 0/320 [00:00<?, ?it/s]


PIXEL DOMAIN SUMMARY
           raw_mean                               raw_std                           display_mean_0_255                        display_std_0_255                      near_black_fraction                     near_white_fraction                    
               mean       median         std         mean       median          std               mean      median        std              mean     median       std                mean    median       std                mean    median       std
source                                                                                                                                                                                                                                              
IN-CXR   108.096288   110.562847    15.96678    67.155890    69.077763     8.297204         114.009881  115.694322  18.008073         73.078599  74.006427  7.645906            0.146468  0.140147  0.091194            0.017897  0.017728  0.00275

In [4]:
# ============================================================
# CELL 29.3.1 — COMMON PREPROCESSING COMPARISON
# IN-CXR vs VinDr
# ============================================================

import numpy as np
import pandas as pd
import pydicom
import json

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from pydicom.pixel_data_handlers.util import (
    apply_modality_lut,
    apply_voi_lut
)

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_preprocessing_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

SAMPLES_PER_GROUP = 40
RANDOM_STATE = 42

print("=" * 70)
print("LUNGVISION AI — COMMON PREPROCESSING COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# 1. STRATIFIED SAMPLE
# ------------------------------------------------------------

samples = []

for _, group in manifest.groupby(
    ["source", "split", "label"]
):
    samples.append(
        group.sample(
            n=min(SAMPLES_PER_GROUP, len(group)),
            random_state=RANDOM_STATE
        )
    )

sample_df = pd.concat(samples, ignore_index=True)

print("Sample count:", len(sample_df))

# ------------------------------------------------------------
# 2. NORMALIZATION HELPERS
# ------------------------------------------------------------

def percentile_normalize(array, low=1, high=99):
    array = np.asarray(array, dtype=np.float32)

    lo, hi = np.percentile(array, [low, high])

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(array, dtype=np.float32)

    return np.clip((array - lo) / (hi - lo), 0, 1)


def normalize_uint8(array):
    return np.round(
        np.clip(array, 0, 1) * 255
    ).astype(np.uint8)


# ------------------------------------------------------------
# 3. IMAGE DECODING
# ------------------------------------------------------------

def load_incxr(path):
    with Image.open(path) as image:
        array = np.asarray(
            image.convert("L"),
            dtype=np.float32
        )

    return array


def load_vindr(path):
    ds = pydicom.dcmread(str(path), force=True)
    raw = ds.pixel_array.astype(np.float32)

    try:
        modality = apply_modality_lut(raw, ds).astype(np.float32)
    except Exception:
        modality = raw.copy()

    has_voi = (
        hasattr(ds, "VOILUTSequence") or
        (
            hasattr(ds, "WindowCenter") and
            hasattr(ds, "WindowWidth")
        )
    )

    if has_voi:
        try:
            display = apply_voi_lut(
                modality,
                ds,
                index=0
            ).astype(np.float32)
        except Exception:
            display = modality.copy()
    else:
        display = modality.copy()

    if str(
        getattr(ds, "PhotometricInterpretation", "")
    ) == "MONOCHROME1":
        display = np.max(display) - display

    return display


# ------------------------------------------------------------
# 4. PROCESSING COMPARISON
# ------------------------------------------------------------

records = []

for row in tqdm(
    sample_df.itertuples(index=False),
    total=len(sample_df),
    desc="Preprocessing comparison"
):

    try:
        if row.source == "IN-CXR":
            raw = load_incxr(row.file_path)
        else:
            raw = load_vindr(row.file_path)

        # Method A: Per-image percentile normalization
        per_image = percentile_normalize(raw)

        # Method B: Common clipping based on each image's
        # intensity percentiles, followed by [0,1] scaling.
        common = percentile_normalize(raw, 1, 99)

        # Method C: Min-max normalization
        minimum = np.min(raw)
        maximum = np.max(raw)

        if maximum > minimum:
            minmax = (raw - minimum) / (maximum - minimum)
        else:
            minmax = np.zeros_like(raw)

        methods = {
            "percentile_1_99": per_image,
            "common_percentile": common,
            "minmax": minmax
        }

        for method_name, image in methods.items():

            records.append({
                "source": row.source,
                "split": row.split,
                "label": row.label_name,
                "image_id": row.image_id,
                "method": method_name,
                "mean_0_255": float(image.mean() * 255),
                "std_0_255": float(image.std() * 255),
                "near_black_fraction": float(
                    np.mean(image <= 0.02)
                ),
                "near_white_fraction": float(
                    np.mean(image >= 0.98)
                )
            })

    except Exception as error:
        print(
            f"Failed: {row.image_id} | {error}"
        )

comparison_df = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. SOURCE-LEVEL COMPARISON
# ------------------------------------------------------------

summary = (
    comparison_df.groupby(["method", "source"])[
        [
            "mean_0_255",
            "std_0_255",
            "near_black_fraction",
            "near_white_fraction"
        ]
    ]
    .agg(["mean", "median", "std"])
)

print("\nSOURCE-LEVEL PREPROCESSING SUMMARY")
print(summary.to_string())

# ------------------------------------------------------------
# 6. SOURCE INTENSITY DIFFERENCE
# ------------------------------------------------------------

difference_records = []

for method, group in comparison_df.groupby("method"):

    means = group.groupby("source")["mean_0_255"].mean()

    incxr_mean = float(means.get("IN-CXR", np.nan))
    vindr_mean = float(means.get("VinDr", np.nan))

    difference_records.append({
        "method": method,
        "incxr_mean": incxr_mean,
        "vindr_mean": vindr_mean,
        "absolute_source_difference": abs(
            incxr_mean - vindr_mean
        )
    })

difference_df = pd.DataFrame(difference_records)

print("\nSOURCE INTENSITY DIFFERENCE")
print(difference_df.to_string(index=False))

# ------------------------------------------------------------
# 7. LABEL-LEVEL COMPARISON
# ------------------------------------------------------------

label_summary = (
    comparison_df.groupby(
        ["method", "source", "label"]
    )[
        ["mean_0_255", "std_0_255"]
    ]
    .agg(["mean", "median"])
)

print("\nLABEL-LEVEL SUMMARY")
print(label_summary.to_string())

# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

comparison_df.to_csv(
    OUTPUT_DIR / "preprocessing_comparison_metrics.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "preprocessing_source_summary.csv"
)

difference_df.to_csv(
    OUTPUT_DIR / "source_intensity_difference.csv",
    index=False
)

label_summary.to_csv(
    OUTPUT_DIR / "preprocessing_label_summary.csv"
)

metadata = {
    "sample_count": int(len(sample_df)),
    "successful_processed_records": int(len(comparison_df)),
    "methods": [
        "percentile_1_99",
        "common_percentile",
        "minmax"
    ],
    "random_state": RANDOM_STATE,
    "interpretation": (
        "This experiment compares per-image intensity "
        "normalization methods. It does not prove that "
        "any method removes source-specific information."
    )
}

with open(
    OUTPUT_DIR / "preprocessing_comparison_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("CELL 29.3.1 COMPLETED")
print("=" * 70)

print("Output directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

LUNGVISION AI — COMMON PREPROCESSING COMPARISON
Sample count: 320


Preprocessing comparison:   0%|          | 0/320 [00:00<?, ?it/s]


SOURCE-LEVEL PREPROCESSING SUMMARY
                          mean_0_255                         std_0_255                       near_black_fraction                     near_white_fraction                    
                                mean      median        std       mean     median        std                mean    median       std                mean    median       std
method            source                                                                                                                                                    
common_percentile IN-CXR  114.009881  115.694322  18.008073  73.078599  74.006427   7.645906            0.146468  0.140147  0.091194            0.017897  0.017728  0.002757
                  VinDr   157.494399  159.719154  18.360631  72.377264  71.961569   9.432526            0.061834  0.039629  0.053764            0.037883  0.028048  0.026266
minmax            IN-CXR  108.232696  109.767632  16.784785  68.132309  69.899174   8.250650       

In [5]:
# ============================================================
# CELL 29.3.2 — ASPECT RATIO AND PADDING AUDIT
# ============================================================

import numpy as np
import pandas as pd
import pydicom
import json

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from pydicom.pixel_data_handlers.util import (
    apply_modality_lut,
    apply_voi_lut
)

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_padding_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 224
SAMPLES_PER_GROUP = 40
RANDOM_STATE = 42

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

print("=" * 70)
print("LUNGVISION AI — ASPECT RATIO AND PADDING AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. STRATIFIED SAMPLE
# ------------------------------------------------------------

samples = []

for _, group in manifest.groupby(["source", "split", "label"]):
    samples.append(
        group.sample(
            n=min(SAMPLES_PER_GROUP, len(group)),
            random_state=RANDOM_STATE
        )
    )

sample_df = pd.concat(samples, ignore_index=True)

# ------------------------------------------------------------
# 2. IMAGE DECODING
# ------------------------------------------------------------

def normalize_image(array):
    array = np.asarray(array, dtype=np.float32)

    lo, hi = np.percentile(array, [1, 99])

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(array, dtype=np.float32)

    return np.clip((array - lo) / (hi - lo), 0, 1)


def load_image(row):
    if row.source == "IN-CXR":
        with Image.open(row.file_path) as image:
            return np.asarray(
                image.convert("L"),
                dtype=np.float32
            )

    ds = pydicom.dcmread(row.file_path, force=True)
    raw = ds.pixel_array.astype(np.float32)

    try:
        modality = apply_modality_lut(raw, ds).astype(np.float32)
    except Exception:
        modality = raw.copy()

    has_voi = (
        hasattr(ds, "VOILUTSequence") or
        (
            hasattr(ds, "WindowCenter") and
            hasattr(ds, "WindowWidth")
        )
    )

    if has_voi:
        try:
            display = apply_voi_lut(
                modality, ds, index=0
            ).astype(np.float32)
        except Exception:
            display = modality.copy()
    else:
        display = modality.copy()

    if str(
        getattr(ds, "PhotometricInterpretation", "")
    ) == "MONOCHROME1":
        display = np.max(display) - display

    return display


# ------------------------------------------------------------
# 3. RESIZING METHODS
# ------------------------------------------------------------

def resize_image(array, width, height):
    image = Image.fromarray(
        np.round(np.clip(array, 0, 1) * 255).astype(np.uint8)
    )

    resized = image.resize(
        (width, height),
        Image.Resampling.BILINEAR
    )

    return np.asarray(resized, dtype=np.float32) / 255.0


def corner_median(array):
    h, w = array.shape

    ch = max(1, int(h * 0.05))
    cw = max(1, int(w * 0.05))

    corners = np.concatenate([
        array[:ch, :cw].ravel(),
        array[:ch, -cw:].ravel(),
        array[-ch:, :cw].ravel(),
        array[-ch:, -cw:].ravel()
    ])

    return float(np.median(corners))


def aspect_preserving_resize(array, mode):
    h, w = array.shape

    scale = min(
        TARGET_SIZE / w,
        TARGET_SIZE / h
    )

    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))

    resized = resize_image(array, new_w, new_h)

    pad_w = TARGET_SIZE - new_w
    pad_h = TARGET_SIZE - new_h

    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top

    if mode == "reflect":
        padded = np.pad(
            resized,
            ((top, bottom), (left, right)),
            mode="reflect"
        )

    elif mode == "zero":
        padded = np.pad(
            resized,
            ((top, bottom), (left, right)),
            mode="constant",
            constant_values=0
        )

    elif mode == "corner_median":
        value = corner_median(resized)

        padded = np.pad(
            resized,
            ((top, bottom), (left, right)),
            mode="constant",
            constant_values=value
        )

    else:
        raise ValueError(mode)

    return padded, new_w, new_h


# ------------------------------------------------------------
# 4. AUDIT METHODS
# ------------------------------------------------------------

records = []

for row in tqdm(
    sample_df.itertuples(index=False),
    total=len(sample_df),
    desc="Padding audit"
):

    try:
        raw = load_image(row)
        normalized = normalize_image(raw)

        h, w = normalized.shape

        # Direct resize
        direct = resize_image(
            normalized,
            TARGET_SIZE,
            TARGET_SIZE
        )

        direct_width = TARGET_SIZE
        direct_height = TARGET_SIZE

        # Aspect-preserving methods
        for method in ["reflect", "zero", "corner_median"]:

            processed, new_w, new_h = aspect_preserving_resize(
                normalized,
                method
            )

            pad_fraction = (
                1 - (new_w * new_h) /
                (TARGET_SIZE * TARGET_SIZE)
            )

            records.append({
                "source": row.source,
                "split": row.split,
                "label": row.label_name,
                "image_id": row.image_id,
                "method": method,
                "original_width": w,
                "original_height": h,
                "resized_width": new_w,
                "resized_height": new_h,
                "pad_fraction": float(pad_fraction),
                "mean_intensity": float(processed.mean() * 255),
                "std_intensity": float(processed.std() * 255),
                "near_black_fraction": float(
                    np.mean(processed <= 0.02)
                )
            })

        records.append({
            "source": row.source,
            "split": row.split,
            "label": row.label_name,
            "image_id": row.image_id,
            "method": "direct_stretch",
            "original_width": w,
            "original_height": h,
            "resized_width": direct_width,
            "resized_height": direct_height,
            "pad_fraction": 0.0,
            "mean_intensity": float(direct.mean() * 255),
            "std_intensity": float(direct.std() * 255),
            "near_black_fraction": float(
                np.mean(direct <= 0.02)
            )
        })

    except Exception as error:
        print(f"Failed: {row.image_id}: {error}")

padding_df = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

summary = (
    padding_df.groupby(["source", "method"])[
        [
            "pad_fraction",
            "mean_intensity",
            "std_intensity",
            "near_black_fraction"
        ]
    ]
    .agg(["mean", "median", "std"])
)

print("\nPADDING SUMMARY")
print(summary.to_string())

# ------------------------------------------------------------
# 6. ORIGINAL ASPECT RATIO
# ------------------------------------------------------------

padding_df["original_aspect_ratio"] = (
    padding_df["original_width"] /
    padding_df["original_height"]
)

aspect_summary = (
    padding_df.groupby("source")["original_aspect_ratio"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

print("\nORIGINAL ASPECT RATIO")
print(aspect_summary.to_string())

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

padding_df.to_csv(
    OUTPUT_DIR / "padding_audit_metrics.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "padding_method_summary.csv"
)

aspect_summary.to_csv(
    OUTPUT_DIR / "aspect_ratio_summary.csv"
)

metadata = {
    "selected_images": int(len(sample_df)),
    "processed_records": int(len(padding_df)),
    "target_size": TARGET_SIZE,
    "methods": [
        "direct_stretch",
        "reflect",
        "zero",
        "corner_median"
    ],
    "random_state": RANDOM_STATE,
    "limitations": [
        "Sample-based preprocessing comparison.",
        "Padding statistics do not establish diagnostic superiority.",
        "No image or dataset files were modified."
    ]
}

with open(
    OUTPUT_DIR / "padding_audit_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("CELL 29.3.2 COMPLETED")
print("=" * 70)

print("Output directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

LUNGVISION AI — ASPECT RATIO AND PADDING AUDIT


Padding audit:   0%|          | 0/320 [00:00<?, ?it/s]


PADDING SUMMARY
                      pad_fraction                     mean_intensity                        std_intensity                       near_black_fraction                    
                              mean    median       std           mean      median        std          mean     median        std                mean    median       std
source method                                                                                                                                                           
IN-CXR corner_median      0.000000  0.000000  0.000000     114.005860  115.685471  18.012343     73.089198  74.022495   7.649296            0.147118  0.140824  0.091453
       direct_stretch     0.000000  0.000000  0.000000     114.005860  115.685471  18.012343     73.089198  74.022495   7.649296            0.147118  0.140824  0.091453
       reflect            0.000000  0.000000  0.000000     114.005860  115.685471  18.012343     73.089198  74.022495   7.649296          

In [6]:
import pandas as pd
from pathlib import Path

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_clean_manifest.csv"
)

print("=" * 65)
print("IN-CXR AUDIT FAILURE DIAGNOSTICS")
print("=" * 65)

print("\nManifest columns:")
print(manifest.columns.tolist())

print("\nManifest shape:")
print(manifest.shape)

incxr = manifest[manifest["source"] == "IN-CXR"].copy()

print("\nIN-CXR records:", len(incxr))

print("\nFirst record:")
print(incxr.iloc[0].to_dict())

print("\nFile path check:")
path = Path(str(incxr.iloc[0]["file_path"]))

print("Path:", path)
print("Exists:", path.exists())

print("\nRequired columns:")
for col in ["image_id", "label_name", "split", "file_path"]:
    print(col, ":", col in incxr.columns)

print("\nPIL decoding test:")

from PIL import Image
import numpy as np

try:
    with Image.open(path) as image:
        array = np.asarray(image.convert("L"), dtype=np.float32)

    print("SUCCESS")
    print("Shape:", array.shape)
    print("Mean:", array.mean())

except Exception as e:
    print("FAILED:", repr(e))

print("\nPrevious error list:")
if "errors" in globals():
    print("Error count:", len(errors))
    if errors:
        print("First five errors:")
        print(pd.DataFrame(errors).head().to_string(index=False))
else:
    print("No previous error list available.")

IN-CXR AUDIT FAILURE DIAGNOSTICS

Manifest columns:
['image_id', 'source', 'label', 'label_name', 'file_path', 'file_name', 'extension', 'patient_id', 'reader_count', 'reader_unanimous', 'sha256', 'exact_duplicate', 'duplicate_group', 'quarantined']

Manifest shape:
(27514, 14)

IN-CXR records: 12514

First record:
{'image_id': '0002cf17990411884714c891625b41f8_Abnormal', 'source': 'IN-CXR', 'label': 1, 'label_name': 'Abnormal', 'file_path': '/kaggle/input/datasets/arjav007/in-cxr-dataset-png/IN-CXR (pre-processed)/AbNormal/0002cf17990411884714c891625b41f8_Abnormal.png', 'file_name': '0002cf17990411884714c891625b41f8_Abnormal.png', 'extension': '.png', 'patient_id': nan, 'reader_count': nan, 'reader_unanimous': nan, 'sha256': '6d463437916ee5923731c7aac3489f21a437f1f1019aea3e04d5d729d5aa481a', 'exact_duplicate': False, 'duplicate_group': nan, 'quarantined': False}

File path check:
Path: /kaggle/input/datasets/arjav007/in-cxr-dataset-png/IN-CXR (pre-processed)/AbNormal/0002cf17990411884

In [7]:
# ============================================================
# CELL 29.3.3 — FULL IN-CXR ARTIFACT AUDIT (FIXED)
# ============================================================

import numpy as np
import pandas as pd
import json

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_incxr_artifact_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

incxr = manifest[
    manifest["source"] == "IN-CXR"
].copy()

print("=" * 70)
print("LUNGVISION AI — FULL IN-CXR ARTIFACT AUDIT")
print("=" * 70)

print("Images to audit:", len(incxr))

records = []
errors = []

for _, row in tqdm(
    incxr.iterrows(),
    total=len(incxr),
    desc="IN-CXR artifact audit"
):
    try:
        with Image.open(row["file_path"]) as image:
            array = np.asarray(
                image.convert("L"),
                dtype=np.float32
            )

        near_black = float(np.mean(array <= 10))
        near_white = float(np.mean(array >= 245))

        records.append({
            "image_id": row["image_id"],
            "label_name": row["label_name"],
            "split": row["split"],
            "width": int(array.shape[1]),
            "height": int(array.shape[0]),
            "mean_intensity": float(array.mean()),
            "std_intensity": float(array.std()),
            "minimum": float(array.min()),
            "maximum": float(array.max()),
            "near_black_fraction": near_black,
            "near_white_fraction": near_white,
            "dark_region_flag": near_black >= 0.30
        })

    except Exception as error:
        errors.append({
            "image_id": row["image_id"],
            "file_path": row["file_path"],
            "error": repr(error)
        })

# ------------------------------------------------------------
# 1. CREATE AND SAVE RAW RESULTS
# ------------------------------------------------------------

metrics = pd.DataFrame(records)
error_df = pd.DataFrame(errors)

if metrics.empty:
    raise RuntimeError(
        "No images decoded. Inspect error_df before continuing."
    )

metrics.to_csv(
    OUTPUT_DIR / "incxr_full_artifact_metrics.csv",
    index=False
)

error_df.to_csv(
    OUTPUT_DIR / "incxr_artifact_errors.csv",
    index=False
)

print("\nDECODE RESULTS")
print("Expected:", len(incxr))
print("Decoded:", len(metrics))
print("Failed:", len(error_df))

# ------------------------------------------------------------
# 2. LABEL SUMMARY
# ------------------------------------------------------------

label_summary = (
    metrics.groupby("label_name")[
        [
            "mean_intensity",
            "std_intensity",
            "near_black_fraction",
            "near_white_fraction"
        ]
    ]
    .agg(["mean", "median", "std"])
)

print("\nAUDIT SUMMARY BY LABEL")
print(label_summary.to_string())

# ------------------------------------------------------------
# 3. SPLIT AND LABEL SUMMARY
# ------------------------------------------------------------

dark_summary = (
    metrics.groupby(["split", "label_name"])
    .agg(
        total_images=("image_id", "count"),
        flagged_images=("dark_region_flag", "sum")
    )
    .reset_index()
)

dark_summary["flagged_percentage"] = (
    dark_summary["flagged_images"] /
    dark_summary["total_images"] * 100
)

print("\nDARK REGION FLAG SUMMARY")
print(dark_summary.to_string(index=False))

# ------------------------------------------------------------
# 4. IMAGE DIMENSION CHECK
# ------------------------------------------------------------

print("\nIMAGE DIMENSIONS")
print(metrics.groupby(["width", "height"]).size())

# ------------------------------------------------------------
# 5. SAVE SUMMARIES
# ------------------------------------------------------------

label_summary.to_csv(
    OUTPUT_DIR / "incxr_artifact_summary_by_label.csv"
)

dark_summary.to_csv(
    OUTPUT_DIR / "incxr_dark_region_flag_summary.csv",
    index=False
)

metadata = {
    "expected_images": int(len(incxr)),
    "decoded_images": int(len(metrics)),
    "failed_images": int(len(error_df)),
    "dark_region_threshold": 0.30,
    "dark_region_flag_is_heuristic": True,
    "images_modified": False,
    "images_deleted": False,
    "limitations": [
        "Dark-region flags do not establish image corruption.",
        "Legitimate radiographic anatomy can produce dark regions.",
        "No automatic artifact removal was performed."
    ]
}

with open(
    OUTPUT_DIR / "incxr_artifact_audit_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

# ------------------------------------------------------------
# 6. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 29.3.3 COMPLETED")
print("=" * 70)

print("Expected images:", len(incxr))
print("Successfully decoded:", len(metrics))
print("Failed:", len(error_df))

print("\nOutput directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(file.name)

LUNGVISION AI — FULL IN-CXR ARTIFACT AUDIT
Images to audit: 12514


IN-CXR artifact audit:   0%|          | 0/12514 [00:00<?, ?it/s]


DECODE RESULTS
Expected: 12514
Decoded: 12514
Failed: 0

AUDIT SUMMARY BY LABEL
           mean_intensity                        std_intensity                      near_black_fraction                     near_white_fraction                    
                     mean      median        std          mean     median       std                mean    median       std                mean    median       std
label_name                                                                                                                                                         
Abnormal       107.869932  110.238392  17.902287     67.333062  69.331490  9.041156            0.157305  0.147242  0.108792            0.003896  0.003528  0.002525
Normal         106.711964  108.996391  15.081405     68.083938  69.492455  7.471208            0.151943  0.144790  0.089821            0.003734  0.003508  0.002475

DARK REGION FLAG SUMMARY
     split label_name  total_images  flagged_images  flagged_percentage
 

In [8]:
# ============================================================
# CELL 29.4 — MULTI-SOURCE RESNET18 BASELINE
# IN-CXR + VinDr | Binary Classification
# ============================================================

import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.models import ResNet18_Weights
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from pydicom.pixel_data_handlers.util import (
    apply_modality_lut,
    apply_voi_lut
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 2
NUM_WORKERS = 2
IMAGE_SIZE = 224

AUDIT_DIR = Path("/kaggle/working/lungvision_phase2_data_audit")
OUTPUT_DIR = Path("/kaggle/working/lungvision_phase2_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("LUNGVISION AI — MULTI-SOURCE RESNET18 BASELINE")
print("=" * 70)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

# ------------------------------------------------------------
# 2. LOAD FROZEN DATA SPLIT
# ------------------------------------------------------------

split_df = pd.read_csv(
    AUDIT_DIR / "phase2_train_validation_split.csv"
)

required = {
    "source",
    "label",
    "label_name",
    "split",
    "file_path",
    "image_id"
}

missing = required - set(split_df.columns)

if missing:
    raise ValueError(f"Missing manifest columns: {missing}")

train_df = split_df[
    split_df["split"] == "train"
].copy().reset_index(drop=True)

val_df = split_df[
    split_df["split"] == "validation"
].copy().reset_index(drop=True)

assert len(train_df) > 0 and len(val_df) > 0

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining source/class distribution:")
print(pd.crosstab(train_df["source"], train_df["label_name"]))

print("\nValidation source/class distribution:")
print(pd.crosstab(val_df["source"], val_df["label_name"]))

# ------------------------------------------------------------
# 3. IMAGE PREPROCESSING
# ------------------------------------------------------------

def percentile_normalize(array):
    array = np.asarray(array, dtype=np.float32)

    low, high = np.percentile(array, [1, 99])

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return np.zeros_like(array, dtype=np.float32)

    return np.clip(
        (array - low) / (high - low),
        0,
        1
    )


def corner_median(array):
    h, w = array.shape

    ch = max(1, int(h * 0.05))
    cw = max(1, int(w * 0.05))

    corners = np.concatenate([
        array[:ch, :cw].ravel(),
        array[:ch, -cw:].ravel(),
        array[-ch:, :cw].ravel(),
        array[-ch:, -cw:].ravel()
    ])

    return float(np.median(corners))


def resize_and_pad(array):
    h, w = array.shape

    scale = min(
        IMAGE_SIZE / w,
        IMAGE_SIZE / h
    )

    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))

    image = Image.fromarray(
        np.round(np.clip(array, 0, 1) * 255).astype(np.uint8)
    )

    image = image.resize(
        (new_w, new_h),
        Image.Resampling.BILINEAR
    )

    resized = np.asarray(
        image,
        dtype=np.float32
    ) / 255.0

    pad_w = IMAGE_SIZE - new_w
    pad_h = IMAGE_SIZE - new_h

    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top

    fill = corner_median(resized)

    return np.pad(
        resized,
        ((top, bottom), (left, right)),
        mode="constant",
        constant_values=fill
    )


def load_incxr(path):
    with Image.open(path) as image:
        array = np.asarray(
            image.convert("L"),
            dtype=np.float32
        )

    if array.shape == (IMAGE_SIZE, IMAGE_SIZE):
        return percentile_normalize(array)

    return resize_and_pad(percentile_normalize(array))


def load_vindr(path):
    ds = pydicom.dcmread(str(path), force=True)

    raw = ds.pixel_array.astype(np.float32)

    try:
        modality = apply_modality_lut(
            raw,
            ds
        ).astype(np.float32)
    except Exception:
        modality = raw.copy()

    has_voi = (
        hasattr(ds, "VOILUTSequence") or
        (
            hasattr(ds, "WindowCenter") and
            hasattr(ds, "WindowWidth")
        )
    )

    if has_voi:
        try:
            display = apply_voi_lut(
                modality,
                ds,
                index=0
            ).astype(np.float32)
        except Exception:
            display = modality.copy()
    else:
        display = modality.copy()

    if str(
        getattr(ds, "PhotometricInterpretation", "")
    ) == "MONOCHROME1":
        display = np.max(display) - display

    return resize_and_pad(
        percentile_normalize(display)
    )


# ------------------------------------------------------------
# 4. PYTORCH DATASET
# ------------------------------------------------------------

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
).view(3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
).view(3, 1, 1)


class LungVisionDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        if row["source"] == "IN-CXR":
            image = load_incxr(row["file_path"])
        elif row["source"] == "VinDr":
            image = load_vindr(row["file_path"])
        else:
            raise ValueError(
                f"Unknown source: {row['source']}"
            )

        tensor = torch.from_numpy(
            image.astype(np.float32)
        ).unsqueeze(0)

        tensor = tensor.repeat(3, 1, 1)

        tensor = (
            tensor - IMAGENET_MEAN
        ) / IMAGENET_STD

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return tensor, label


train_dataset = LungVisionDataset(train_df)
val_dataset = LungVisionDataset(val_df)

# ------------------------------------------------------------
# 5. BALANCED SOURCE-CLASS SAMPLING
# ------------------------------------------------------------

group_keys = (
    train_df["source"].astype(str) + "_" +
    train_df["label"].astype(str)
)

group_counts = group_keys.value_counts()

sample_weights = group_keys.map(
    lambda key: 1.0 / group_counts[key]
).to_numpy(dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights),
    num_samples=len(train_df),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

print("\nBalanced sampling groups:")
print(group_counts)

# ------------------------------------------------------------
# 6. MODEL INITIALIZATION
# ------------------------------------------------------------

print("\nLoading ImageNet-pretrained ResNet18...")

try:
    model = models.resnet18(
        weights=ResNet18_Weights.DEFAULT
    )
except Exception as error:
    raise RuntimeError(
        "Unable to load ImageNet pretrained weights. "
        "Check Kaggle internet access or torchvision weight cache. "
        "Do not silently continue with random initialization."
    ) from error

model.fc = nn.Linear(
    model.fc.in_features,
    1
)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 7. EVALUATION FUNCTION
# ------------------------------------------------------------

def calculate_metrics(y_true, probabilities):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": float(
            accuracy_score(y_true, predictions)
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "specificity": float(
            tn / (tn + fp) if (tn + fp) else 0
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "roc_auc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def evaluate_model():

    model.eval()

    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in tqdm(
            val_loader,
            desc="Validation",
            leave=False
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):
                logits = model(images).squeeze(1)

            probabilities = torch.sigmoid(logits)

            all_labels.extend(labels.numpy().tolist())
            all_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

    predictions_df = val_df.copy()

    predictions_df["probability"] = all_probabilities
    predictions_df["prediction"] = (
        predictions_df["probability"] >= 0.5
    ).astype(int)

    results = {}

    for source in ["IN-CXR", "VinDr"]:

        subset = predictions_df[
            predictions_df["source"] == source
        ]

        results[source] = calculate_metrics(
            subset["label"].to_numpy(),
            subset["probability"].to_numpy()
        )

    results["POOLED"] = calculate_metrics(
        predictions_df["label"].to_numpy(),
        predictions_df["probability"].to_numpy()
    )

    return results, predictions_df


# ------------------------------------------------------------
# 8. TRAINING LOOP WITH CHECKPOINTING
# ------------------------------------------------------------

history = []
best_macro_auc = -1
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):

    print("\n" + "=" * 70)
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print("=" * 70)

    model.train()

    running_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Training epoch {epoch}"
    )

    for images, labels in progress:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / total_samples

    # Evaluate after each epoch
    epoch_metrics, predictions_df = evaluate_model()

    macro_auc = (
        epoch_metrics["IN-CXR"]["roc_auc"] +
        epoch_metrics["VinDr"]["roc_auc"]
    ) / 2

    scheduler.step(macro_auc)

    print("\nTraining loss:", round(train_loss, 5))
    print("Macro source AUC:", round(macro_auc, 5))

    for source, metrics in epoch_metrics.items():
        print(f"\n{source}")

        for metric, value in metrics.items():
            print(f"{metric}: {value}")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "macro_source_auc": macro_auc,
        "incxr_auc": epoch_metrics["IN-CXR"]["roc_auc"],
        "vindr_auc": epoch_metrics["VinDr"]["roc_auc"],
        "pooled_auc": epoch_metrics["POOLED"]["roc_auc"],
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    # Save predictions every epoch
    predictions_df.to_csv(
        OUTPUT_DIR / f"validation_predictions_epoch_{epoch}.csv",
        index=False
    )

    # Save epoch checkpoint
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "validation_metrics": epoch_metrics,
            "macro_source_auc": macro_auc,
            "seed": SEED
        },
        OUTPUT_DIR / f"resnet18_epoch_{epoch}.pth"
    )

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False
    )

    # Best model based on equal-weight source AUC
    if macro_auc > best_macro_auc:

        best_macro_auc = macro_auc
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "validation_metrics": epoch_metrics,
                "macro_source_auc": macro_auc,
                "seed": SEED
            },
            OUTPUT_DIR / "resnet18_best.pth"
        )

        print("\nBest model checkpoint updated.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

metadata = {
    "model": "ResNet18",
    "pretrained_weights": "ImageNet DEFAULT",
    "task": "Binary Normal vs Abnormal classification",
    "input_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "completed_epochs": len(history),
    "best_epoch": best_epoch,
    "best_macro_source_auc": float(best_macro_auc),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "random_seed": SEED,
    "patient_disjoint_validation_certified": False,
    "output_directory": str(OUTPUT_DIR)
}

with open(
    OUTPUT_DIR / "training_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("CELL 29.4 — TRAINING COMPLETED")
print("=" * 70)

print("Best epoch:", best_epoch)
print("Best macro source AUC:", round(best_macro_auc, 5))
print("Output directory:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(
            file.name,
            f"({file.stat().st_size / 1024**2:.2f} MB)"
        )

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

LUNGVISION AI — MULTI-SOURCE RESNET18 BASELINE
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB

Training images: 22011
Validation images: 5503

Training source/class distribution:
label_name  Abnormal  Normal
source                      
IN-CXR          4976    5035
VinDr           3515    8485

Validation source/class distribution:
label_name  Abnormal  Normal
source                      
IN-CXR          1244    1259
VinDr            879    2121

Balanced sampling groups:
VinDr_0     8485
IN-CXR_0    5035
IN-CXR_1    4976
VinDr_1     3515
Name: count, dtype: int64

Loading ImageNet-pretrained ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]



EPOCH 1/5


Training epoch 1:   0%|          | 0/344 [00:00<?, ?it/s]

Validation:   0%|          | 0/86 [00:00<?, ?it/s]


Training loss: 0.29965
Macro source AUC: 0.93501

IN-CXR
accuracy: 0.7610866959648422
precision: 0.701875
recall: 0.9027331189710611
specificity: 0.6211278792692613
f1: 0.789732770745429
roc_auc: 0.8948298935765383
average_precision: 0.9040850684788762
tn: 782
fp: 477
fn: 121
tp: 1123

VinDr
accuracy: 0.9266666666666666
precision: 0.8577633007600435
recall: 0.8987485779294653
specificity: 0.9382366808109383
f1: 0.8777777777777778
roc_auc: 0.9751842322213693
average_precision: 0.9508802731725029
tn: 1990
fp: 131
fn: 89
tp: 790

POOLED
accuracy: 0.8513538070143558
precision: 0.7588258627528759
recall: 0.9010833725859633
specificity: 0.8201183431952662
f1: 0.8238587424633936
roc_auc: 0.9461697330170825
average_precision: 0.9228142797233381
tn: 2772
fp: 608
fn: 210
tp: 1913

Best model checkpoint updated.

EPOCH 2/5


... (8 outputs left out)


Training loss: 0.08916
Macro source AUC: 0.94883

IN-CXR
accuracy: 0.8473831402317219
precision: 0.8774080560420315
recall: 0.8054662379421221
specificity: 0.8888006354249405
f1: 0.8398994132439229
roc_auc: 0.9173076039014274
average_precision: 0.9243639928719388
tn: 1119
fp: 140
fn: 242
tp: 1002

VinDr
accuracy: 0.94
precision: 0.9267399267399268
recall: 0.863481228668942
specificity: 0.9717114568599717
f1: 0.8939929328621908
roc_auc: 0.9803570556958183
average_precision: 0.9640690767875278
tn: 2061
fp: 60
fn: 120
tp: 759

POOLED
accuracy: 0.8978738869707432
precision: 0.8980112187659357
recall: 0.8294865756005653
specificity: 0.9408284023668639
f1: 0.8623898139079333
roc_auc: 0.9583911206370352
average_precision: 0.941677947831651
tn: 3180
fp: 200
fn: 362
tp: 1761

EPOCH 5/5


Training epoch 5:   0%|          | 0/344 [00:00<?, ?it/s]

Validation:   0%|          | 0/86 [00:00<?, ?it/s]


Training loss: 0.06976
Macro source AUC: 0.9488

IN-CXR
accuracy: 0.8393927287255294
precision: 0.8799638989169675
recall: 0.7837620578778135
specificity: 0.8943606036536934
f1: 0.8290816326530612
roc_auc: 0.9190283336185253
average_precision: 0.9240818102218397
tn: 1126
fp: 133
fn: 269
tp: 975

VinDr
accuracy: 0.9373333333333334
precision: 0.8877665544332211
recall: 0.8998862343572241
specificity: 0.9528524280999529
f1: 0.8937853107344633
roc_auc: 0.9785754782206646
average_precision: 0.9602249908724017
tn: 2021
fp: 100
fn: 88
tp: 791

POOLED
accuracy: 0.8927857532255133
precision: 0.8834417208604303
recall: 0.8318417333961375
specificity: 0.9310650887573965
f1: 0.8568655992236778
roc_auc: 0.9553687145855341
average_precision: 0.9384346346882981
tn: 3147
fp: 233
fn: 357
tp: 1766

Early stopping triggered.

CELL 29.4 — TRAINING COMPLETED
Best epoch: 3
Best macro source AUC: 0.94996
Output directory: /kaggle/working/lungvision_phase2_models
resnet18_best.pth (128.04 MB)
resnet18_epoch_

In [9]:
# ============================================================
# CELL 29.8C — LUNGVISION RESUME & FINAL MODEL ZIP
# ============================================================

import os
import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/lungvision_final_training"
)

ZIP_PATH = Path(
    "/kaggle/working/LungVision_AI_Resume_Package.zip"
)

CACHE_DIR = OUTPUT_DIR / "cache"

# ------------------------------------------------------------
# 2. REQUIRED FILES FOR RESUMING
# ------------------------------------------------------------

required_files = [
    OUTPUT_DIR / "resume_checkpoint.pth",
    OUTPUT_DIR / "final_training_manifest.csv",
    OUTPUT_DIR / "final_training_history.csv",
    CACHE_DIR / "images_uint8.npy",
    CACHE_DIR / "completed.npy",
    CACHE_DIR / "cache_metadata.json"
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Cannot create resume ZIP. Missing files:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# 3. OPTIONAL FINAL MODEL FILES
# ------------------------------------------------------------

optional_files = [
    OUTPUT_DIR / "lungvision_final_resnet18.pth",
    OUTPUT_DIR / "lungvision_final_resnet18_state_dict.pth",
    OUTPUT_DIR / "final_training_metadata.json",
    OUTPUT_DIR / "cache_failures.csv"
]

files_to_package = required_files + [
    path for path in optional_files if path.exists()
]

# ------------------------------------------------------------
# 4. VERIFY CACHE COMPLETION
# ------------------------------------------------------------

import numpy as np

completed = np.load(
    CACHE_DIR / "completed.npy"
)

cache_metadata = json.load(
    open(CACHE_DIR / "cache_metadata.json")
)

if not cache_metadata.get("complete", False):
    raise RuntimeError("Image cache is not marked complete.")

if not completed.all():
    raise RuntimeError(
        f"Cache incomplete: {completed.sum()}/{len(completed)}"
    )

# ------------------------------------------------------------
# 5. CHECK TRAINING PROGRESS
# ------------------------------------------------------------

checkpoint = torch.load(
    OUTPUT_DIR / "resume_checkpoint.pth",
    map_location="cpu",
    weights_only=False
)

completed_epoch = checkpoint["completed_epoch"]
total_epochs = checkpoint["total_epochs"]

print("Completed epochs:", completed_epoch)
print("Total epochs:", total_epochs)

# ------------------------------------------------------------
# 6. CREATE ZIP
# ------------------------------------------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

checksum_records = []

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    allowZip64=True
) as archive:

    for file_path in files_to_package:

        relative_path = file_path.relative_to(OUTPUT_DIR)

        # Store large NumPy cache without expensive recompression.
        compression = (
            zipfile.ZIP_STORED
            if file_path.suffix in [".npy", ".pth"]
            else zipfile.ZIP_DEFLATED
        )

        archive.write(
            file_path,
            arcname=str(relative_path),
            compress_type=compression
        )

        sha256 = hashlib.sha256()

        with open(file_path, "rb") as f:
            for chunk in iter(
                lambda: f.read(8 * 1024 * 1024),
                b""
            ):
                sha256.update(chunk)

        checksum_records.append({
            "file": str(relative_path),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256.hexdigest()
        })

    package_metadata = {
        "project": "LungVision AI",
        "package_type": "Resume training and final model",
        "created_at": datetime.utcnow().isoformat(),
        "completed_epoch": completed_epoch,
        "total_epochs": total_epochs,
        "image_count": int(len(completed)),
        "cache_complete": True,
        "files": checksum_records
    }

    archive.writestr(
        "package_manifest.json",
        json.dumps(package_metadata, indent=4)
    )

# ------------------------------------------------------------
# 7. VERIFY ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    bad_file = archive.testzip()

    if bad_file is not None:
        raise RuntimeError(
            f"ZIP verification failed: {bad_file}"
        )

    archive_names = archive.namelist()

print("\n" + "=" * 65)
print("LUNGVISION ZIP PACKAGE CREATED")
print("=" * 65)

print("ZIP:", ZIP_PATH)
print("Size:", round(ZIP_PATH.stat().st_size / 1024**3, 3), "GB")
print("Completed epoch:", completed_epoch)
print("Files included:", len(archive_names))

print("\nPACKAGE CONTENTS:")

for name in archive_names:
    print(name)

print("\nZIP integrity: VERIFIED")
print("Package ready for Kaggle Dataset upload.")

FileNotFoundError: Cannot create resume ZIP. Missing files:
/kaggle/working/lungvision_final_training/resume_checkpoint.pth
/kaggle/working/lungvision_final_training/final_training_manifest.csv
/kaggle/working/lungvision_final_training/final_training_history.csv
/kaggle/working/lungvision_final_training/cache/images_uint8.npy
/kaggle/working/lungvision_final_training/cache/completed.npy
/kaggle/working/lungvision_final_training/cache/cache_metadata.json